## 1. Importing the necessary libraries
We first import the necessary dependencies for our analysis as well as our data for further processing.

The required libraries are imported below:

In [ ]:
import json
import nltk
import pickle
import pandas as pd
import wordcloud as wc
import matplotlib.pyplot as plt
import seaborn as sns
import bertopic
import os

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN
from plotly import io as pio
from sentence_transformers import SentenceTransformer

nltk.download('stopwords')

def model_topics(corpus, dates, path_to_data="output", min_cluster_size=3, rerun_embeddings=False, rerun_topic=True):
    embeddings_path = os.path.join(path_to_data, "embeddings.p")
    if not os.path.exists(embeddings_path) or rerun_embeddings:
        sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
        embeddings = sentence_model.encode(corpus, show_progress_bar=True)
        pickle.dump(embeddings, open(embeddings_path, "wb"))
    else:
        embeddings = pickle.load(open(embeddings_path, "rb"))

    topic_model_path = os.path.join(path_to_data, "topic_model.p")
    if not os.path.exists(topic_model_path) or rerun_topic:
        ctfidf_model = bertopic.vectorizers.ClassTfidfTransformer(reduce_frequent_words=True)
        vectorizer_model = CountVectorizer(stop_words="english")
        hdbscan_model = HDBSCAN(
            min_cluster_size=min_cluster_size, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

        topic_model = bertopic.BERTopic(
            hdbscan_model=hdbscan_model, 
            ctfidf_model=ctfidf_model, 
            vectorizer_model=vectorizer_model,
            verbose=True)
        topic_model.fit(corpus, embeddings)
        topic_model.save(topic_model_path)
    else:
        topic_model = bertopic.BERTopic.load(topic_model_path)

    return topic_model, embeddings

We then set local variables to determine which corpus ('safety' or 'ethics') we want to use with what date cutoff. If using the ethics corpus, then we may also specify which conference to use.

In [6]:
# Set this variable to true if only want to process information for the titles but not the astracts. Otherwise, both the title and the abstract will be processed.
use_title_only = False
load_all = True  # If true then load all papers regardless of the below parameters

conference = 'both' # Set this variable to choose which conference proceedings to process. The options are 'aies', 'facct', and 'both'.
field = 'safety'  # Either 'safety' or 'ethics'
year_cutoff = 2022  # 2022
month_cutoff = 11  # 11

assert conference in ['aies', 'facct', 'both'], "Invalid conference option. Choose 'aies', 'facct', or 'both'."
assert field in ['safety', 'ethics'], "Invalid field option. Choose 'safety' or 'ethics'."

Now we load the data:

In [ ]:
# Load both JSON files for the AIES and FAccT proceedings and extract either the title or the abstract and title from the file.
corpus = []
dates = []
titles = []
types = []

if load_all:
    aies = json.load(open('data/ethics/AIES.json', "r", encoding="utf-8"))
    facct = json.load(open('data/ethics/FACCT.json', "r", encoding="utf-8"))
    safety = json.load(open('data/safety/all_safety.json', "r", encoding="utf-8"))
    data = safety + aies + facct
elif field == 'ethics':
    aies = json.load(open('data/ethics/AIES.json', "r", encoding="utf-8"))
    facct = json.load(open('data/ethics/FACCT.json', "r", encoding="utf-8"))
    data = aies + facct if conference == 'both' else aies if conference == 'aies' else facct
else:
    safety = json.load(open('data/safety/all_safety.json', "r", encoding="utf-8"))
    data = safety
    
for paper in data:
    year = int(paper['issued']['date-parts'][0][0])
    month = int(paper['issued']['date-parts'][0][1]) if len(paper['issued']['date-parts'][0]) > 1 else 13

    if year_cutoff and year < year_cutoff:
        continue
    if month_cutoff and year == year_cutoff and month < month_cutoff:
        continue

    titles.append(paper['title'])
    if use_title_only:
        corpus.append(paper['title'])
        types.append("safety" if paper in safety else "ethics")
    elif not use_title_only and 'abstract' in paper:
        corpus.append(paper['title'] + ' ' + paper['abstract'])
        dates.append(year)
        types.append("safety" if paper in safety else "ethics")

print(f"Number of papers: {len(corpus)}")

# Create output folders
os.makedirs('output', exist_ok=True)
os.makedirs(f'output/{field}', exist_ok=True)

## 2. Wordcloud plotting
This section plots wordclouds and token distributions but are otherwise not used in the current set of visualisatoin we are using. We will now define a function that will help us plot the wordcloud for the given text data of each conference proceedings.

In [ ]:
# Plot wordclouds
stemmer = nltk.SnowballStemmer(language='english')
analyzer = TfidfVectorizer().build_analyzer()
def stemmed_words(doc):
    return (stemmer.stem(w.lower()) for w in analyzer(doc) if w not in nltk.corpus.stopwords.words("english"))
tfidf_vectorizer = TfidfVectorizer(analyzer=stemmed_words)
vecs = tfidf_vectorizer.fit_transform(corpus)
tfidf_scores = pd.DataFrame(vecs.todense().tolist(), columns=tfidf_vectorizer.get_feature_names_out()).T.sum(axis=1)

# Save scores to file
tfidf_scores.to_csv(f"output/{field}/tfidf_scores-{conference}.csv")

tfidf_scores.sort_values(ascending=False).head(10)

In [ ]:
wordcloud = wc.WordCloud(width=1000, height=1000, background_color="white", colormap="tab10", random_state=42).generate_from_frequencies(tfidf_scores)
plt.figure(figsize=(10, 10))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.tight_layout()
plt.savefig(f"output/{field}/wordcloud-{conference}.pdf")
plt.show()

In [ ]:
# Plot the top 30 scores from both the AIES and FAccT proceedings on a bar plot.
if field == 'ethics':
    n_scores = 20
    
    aies_scores = pd.read_csv("output/tfidf_scores-aies.csv", index_col=0).reset_index().rename(columns={"index": "Word", "0": "Score"})
    aies_scores["Conference"] = "AIES"
    facct_scores = pd.read_csv("output/tfidf_scores-facct.csv", index_col=0).reset_index().rename(columns={"index": "Word", "0": "Score"})
    facct_scores["Conference"] = "FAccT"
    scores = pd.concat([aies_scores, facct_scores], axis=0).sort_values("Score", ascending=False)
    plt.figure(figsize=(7, 3))
    sns.barplot(data=scores.head(n_scores), x="Word", y="Score", hue="Conference", palette="tab10")
    plt.ylim(10, scores["Score"].max())
    plt.xlabel("Word")
    plt.ylabel("TF-IDF Score")
    plt.title(f"Top {n_scores} TF-IDF Scores")
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.savefig(f"output/tfidf_scores_comparison.pdf")
    plt.show()

## 3. Topic analysis with BERTopic
We will now perform topic analysis on the titles and abstracts of the papers using the BERTopic library.

This uses a utility function from the file `util.py` to embed the data, model topics, and save everything in the appropriate location. Each visualisation will also be saved as an interactive HTML file under the folder `output`.

In [ ]:
import jsonlines

corpus = []
field = "questions"
with jsonlines.open("data/train.jsonl", "r") as reader:
    for i, conversation in enumerate(reader):
        corpus.append(conversation["chosen"])
print(len(corpus))

In [ ]:
# With date limit to 2022: Safety: 8; Ethics: 6
min_cluster_size = 125  # Set this variable to the minimum number of papers in a cluster to be considered a topic. Higher values will result in fewer broader topics while lower values will result in more specific topics.
rerun = False  # Whether to recompute embeddings and the topic model. Otherwise, we will load the existing models, if they exist.
topic_model, embeddings = model_topics(
    corpus, [], path_to_data=f"output/questions", min_cluster_size=min_cluster_size, rerun_embeddings=rerun)

Run the following code snippet to generate a topic and document map:

In [ ]:
reduced_embeddings = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine').fit_transform(embeddings)
fig = topic_model.visualize_documents(corpus, reduced_embeddings=reduced_embeddings)
# fig.write_html(f"output/{field}/documents-{conference}.html", include_mathjax = 'cdn')
fig.show()

Run the following code to visualise which tokens are most frequent under each topic:

In [ ]:
fig = topic_model.visualize_barchart(top_n_topics=12, n_words=6, title=f"{field.capitalize()}: Top 12 Topics and Keywords", custom_labels=True)
# fig.write_html(f"output/{field}/tokens-{conference}.html", include_mathjax = 'cdn')
fig.show()

Run the following code to generate the temporal evolution of topics for the corpus:

In [ ]:
topics_over_time = topic_model.topics_over_time(corpus, dates)
fig = topic_model.visualize_topics_over_time(topics_over_time, topics=list(range(1, 13)))
fig.write_html(f"output/{field}/temporal-{conference}-{min_cluster_size}.html", include_mathjax = 'cdn')
fig.show()

The code below generates a document and topic map with much more pleasing aesthetics, however, it does not provide interactivity:

In [ ]:
import datamapplot

# title = "AIES and FAccT Topics and Documents" if field == 'ethics' else "Safety Topics and Documents"
title = "AI Ethics Topics and Documents since 2022"
fig = topic_model.visualize_document_datamap(corpus, 
                                             title=title, 
                                             sub_title="A visualisation of documents mapped and clustered by topic", 
                                             embeddings=embeddings,
                                             custom_labels=True)
fig.savefig(f"output/{field}/datamap.png", bbox_inches='tight')
fig.show()

# 4. Topic Embedding Comparison

In the following, we take both corpora and model them together to understand which topics are shared and which ones different between the two.

In [ ]:
# Get embeddings and topic model for all data
assert load_all, "Must load all data for subsequent analysis"

output_path = os.path.join("output", "combined")
if not os.path.exists(output_path):
    os.makedirs(output_path)

topic_model, embeddings = model_topics(corpus, dates, output_path, min_cluster_size=8)

In [ ]:
import plotly.express as px
import plotly.graph_objs as go

# Plot documents and topics for all data
reduced_embeddings = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='euclidean').fit_transform(embeddings)

# Convert data to DataFrame for easy plotting

df = pd.DataFrame(reduced_embeddings, columns=["x", "y"])
df["Type"] = [t == "safety" for t in types]
model = topic_model.visualize_documents(corpus, reduced_embeddings=reduced_embeddings)
contour = px.density_contour(df, x="x", y="y", color="Type")
contour.update_traces(hovertemplate=None, hoverlabel=None)
fig = go.Figure(data=contour.data + model.data, layout=model.layout)
fig.write_html(os.path.join(output_path, "documents-combined.html"), include_mathjax = 'cdn')
fig.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import DecisionBoundaryDisplay

X = df.drop(columns="Type").values
y = df["Type"].values
classifier = RandomForestClassifier(n_estimators=200, random_state=42).fit(X, y)
disp = DecisionBoundaryDisplay.from_estimator(
    classifier, X, response_method="predict",
    xlabel="D1", ylabel="D2",
    alpha=0.5,
)   
ax = disp.ax_
scatter = ax.scatter(X[:, 0], X[:, 1], c=y, edgecolor="k", label=types)
legend = ax.legend(*scatter.legend_elements(), loc="lower left", title="Classes")
ax.add_artist(legend)
ax.get_figure().savefig(os.path.join(output_path, "decision_boundary.pdf"))

# 5. Venue Analysis

In this section, we extract and visualise the major venues for AI safety papers. As our AI ethics papers come from FAccT and AIES proceedings, we do not do this for the AI ethics corpus.

In [8]:
import time
import tqdm
import urllib

# Create folder to save the data
output_dir = os.path.join("data", field, "dblp")
os.makedirs(output_dir, exist_ok=True)

# Retrieve publications from DBLP by title
dblp_data = {}
not_found = {}  
raised_error = {}
multiple_hits = {}
try:
    dblp_data = json.load(open(f"{output_dir}/dblp_data.json", "r", encoding="utf-8"))
    not_found = json.load(open(f"{output_dir}/not_found.json", "r", encoding="utf-8"))
    raised_error = json.load(open(f"{output_dir}/raised_error.json", "r", encoding="utf-8"))
    multiple_hits = json.load(open(f"{output_dir}/multiple_hits.json", "r", encoding="utf-8"))
except FileNotFoundError as e:
    print(e)
BASE_API_URL = "https://dblp.org/search/publ/api"

We can now retrieve all publication information from DBLP. Note, these may be incomplete so further processing by hand is required. However, the downloaded data is already located in the data folder so running this cell should not be required.

In [ ]:
for paper in (pbar := tqdm.tqdm(safety)):
    title = paper['title']
    pid = paper['id']

    if pid in dblp_data or pid in not_found or pid in raised_error or pid in multiple_hits:
        continue

    request = f"{BASE_API_URL}?q={title}&format=json"
    request = urllib.parse.quote(request, safe=':/?&=')

    try:
        data = json.load(urllib.request.urlopen(request))
    except urllib.error.HTTPError as e:
        if e.code == 429:
            sleep_time = int(e.headers['Retry-After'])
            print(f"Rate limit exceeded. Sleeping for {sleep_time + 1} seconds.")
            time.sleep(sleep_time + 1.)
            data = json.load(urllib.request.urlopen(request))
        else:
            raised_error.append(pid)
            desc = f"Error retrieving data for {title}: {e}"
    
    hits = int(data['result']['hits']['@total'])
    if hits == 0:
        not_found.append(pid)
        desc = f"Title '{title}' not found"
    elif hits == 1:
        data = data['result']['hits']['hit'][0]
        dblp_data[pid] = data
        desc = f"Retrieved data for '{title}'"
    else:
        not_corr = [hit for hit in data['result']['hits']['hit'] if 'venue' in hit['info'] and hit['info']['venue'] != "CoRR"]
        if len(not_corr) == 1:
            dblp_data[pid] = not_corr[0]
            desc = f"Retrieved data for '{title}'"        
        else:
            desc = f"Multiple hits for '{title}': {hits}"
            multiple_hits[pid] = data

    json.dump(dblp_data, open(os.path.join(output_dir, "dblp_data.json"), "w", encoding="utf-8"), indent=2)
    json.dump(not_found, open(os.path.join(output_dir, "not_found.json"), "w", encoding="utf-8"), indent=2)
    json.dump(raised_error, open(os.path.join(output_dir, "raised_error.json"), "w", encoding="utf-8"), indent=2)
    json.dump(multiple_hits, open(os.path.join(output_dir, "multiple_hits.json"), "w", encoding="utf-8"), indent=2)

    pbar.set_postfix({"result": desc})

    time.sleep(1.)


In [ ]:
# Load DBLP data
dblp_data = json.load(open(os.path.join(output_dir, "dblp_data.json"), "r", encoding="utf-8"))
venues = pd.Series([data['info']['venue'] for data in dblp_data.values()])
counts = venues.value_counts().sort_values(ascending=False)

# Plot the distribution of the top 20 most frequent venues
plt.figure(figsize=(8, 5))
plt.barh(counts.index[:20], counts.values[:20], color="tab:blue")
plt.xlabel("Venue")
plt.ylabel("Frequency")
plt.title("Top 20 Venues")
plt.tight_layout()
plt.savefig(f"output/{field}/venues.pdf")
plt.show()